## Phần 0 — Khởi động (5 phút)

- [ ] **0.1 Seed và device.** Viết hàm `seed_everything(seed)` đặt seed cho `random`, `numpy`, `torch` (CPU + CUDA). Tạo biến `device` tự chọn `cuda` nếu có, không thì `cpu`.
  - **Kiểm tra:** gọi `seed_everything(42)` hai lần, mỗi lần in `torch.rand(3)` → hai kết quả giống hệt nhau.

<details><summary>Gợi ý cú pháp</summary>

`random.seed`, `np.random.seed`, `torch.manual_seed`, `torch.cuda.manual_seed_all`, `torch.cuda.is_available()`

In [29]:
import numpy as np
import torch
import random
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import ToTensor
import torchvision.transforms as T
import albumentations as A 
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import torchvision
import torch.nn as nn
import torch.optim as optim

In [ ]:
def seed_everything(seed: int):
    ''' seed python before
    seed random lib in python
    seed numpy 
    seed torch
    seed cuda 
    seed compute in GPU
    ** all things in python - becuz it is independence
    =====================
    params: seed (int)
    return device - seed '''

    os.environ['PYTHONASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
        torch.cuda.seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(torch.rand((5, 4)))
    print(device)
    return device
    

In [ ]:
seed_everything(42)

**1.1 Đọc nhãn.** Đọc `train.csv` bằng pandas. In số ảnh, số lớp, số ảnh mỗi lớp. Tạo `class2idx` và `idx2class`.

- **Kiểm tra:** `len(class2idx) == 6`; `idx2class[class2idx['glass']] == 'glass'`.

In [ ]:
train_dir = '/Users/quangmanh/Project/Final-competitive/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/annotations.csv'
df = pd.read_csv(train_dir)
print(df.shape)
df.head()

In [ ]:
df_neg = df[df['is_negative'] == False]
df.shape[0] - df_neg.shape[0]

In [ ]:
df['category_id'].value_counts(dropna=False)

In [ ]:
CLASSES = ['no_storm', 'cat1', 'cat2', 'cat3', 'cat4']

In [ ]:
{ idx: cls_name for idx, cls_name in enumerate(CLASSES)}
{ cls_name: idx for idx, cls_name in enumerate(CLASSES)}

In [ ]:
df['label'] = df['category_id'].fillna(0).astype(int)
df.head()

In [ ]:
df['storm_id'] = df['file_name'].str.split('_').str[1]
df['year'] = df['storm_id'].str.slice(0, 2)
df.head()

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, val_idx = next(sgkf.split(df, df.label, groups=df.storm_id))

In [ ]:
train_split, val_split = train_test_split(df, random_state=42, stratify=df.label)

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, df, img_dir, transform = False, has_label=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.has_label = has_label
        self.file_name = self.df.iloc[:,0]
        if self.has_label:
            self.labels = self.df.iloc[:,3]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, index):
        img_name = self.file_name[index]
        img_path = os.join(self.img_dir, img_name)

        image = Image.open(img_path).convert('RGB').resize((224,224))
        image.save(os.join('/Users/quangmanh/Documents/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/images_resize', img_name))

        if self.transform:
            image = self.transform(image)
        else:
            image = ToTensor()(image)
        if self.has_label:
            label = self.labels[index]
            return image, torch.tensor(label, dtype=torch.long)
        return image, img_name

In [ ]:
def make_test_df(img_dir):
    valid_format = ('.png', '.jpg', '.jpeg')

    list_image = sorted(os.listdir(img_dir))

    img_name = [
        f for f in list_image if f.lower().endswith(valid_format) and os.path.isfile(os.path.join(img_dir, f))
    ]

    test_df = pd.DataFrame({'file_name': img_name})
    return test_df

In [ ]:
img_dir = '/Users/quangmanh/Documents/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/images'
train_dataset = ImageDataset(df, img_dir=img_dir, has_label=True)

In [ ]:
dl_tr = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True, pin_memory=True)

In [ ]:
num_sample = df.shape[0]
num_cls = len(df.label.unique())
df_num = df.label.value_counts().sort_index().reset_index()
count_c = (df_num['count']).to_numpy()
print(count_c)
# --------
class_weight = num_sample/(num_cls * count_c)
class_weight.tolist()

In [ ]:
sample_weight = [class_weight[label] for label in df['label']]
sampler = torch.utils.data.WeightedRandomSampler(weights=sample_weight, num_samples=len(sample_weight), replacement=True)

In [ ]:
df['fold'] = -1
stkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(stkf.split(df, df.label, groups=df.storm_id)):
    df.loc[val_idx, 'fold'] = fold_idx
print(df['fold'].value_counts())

In [ ]:
train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop((224, 224), scale=(0.8, 1)),
    T.RandomRotation(180),
    T.ColorJitter(brightness=0.15, contrast=0.1),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
val_transform = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
train_transform_abl = A.Compose([
    A.Resize(256, 256),
    A.RandomResizedCrop((224, 224), scale=(0.8, 1)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.15, contrast=0.1),
    A.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [ ]:
torch.save(torchvision.models.resnet18(weights='DEFAULT').state_dict(), 'resnet_18.pth')

In [7]:
device = torch.device('mps')
load_model = torch.load('resnet_18.pth', map_location=device)
if isinstance(load_model, dict): 
    if 'model_state_dict' in load_model:
        state_dict = load_model['model_state_dict']
        print('object model combination')
    else:
        print('Pure')
        state_dict = load_model
    model = torchvision.models.resnet18(weights=None)
    model.load_state_dict(state_dict)
    print(model)
else:
    model = load_model

model.to(device='mps')

Pure
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=Tru

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [28]:
def build_model(name, num_classes, weight_path):
    model = getattr(torchvision.models, name)(weights=None)
    in_feat = model.fc.in_features
    model.fc = nn.Linear(in_feat, num_classes)
    return model
model = build_model('resnet18', 5, weight_path='resnet_18.pth')
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [24]:
NUM_EPOCH = 5
print(torch.backends.mps.is_available())
device = torch.device('mps')

True


In [25]:
criterion = nn.CrossEntropyLoss()

In [37]:
print(f'Total params of model: {sum(p.numel() for p in model.parameters())}')

print(f'Training loop is starting with: {NUM_EPOCH} epoch')
for epoch in range(NUM_EPOCH):
    if epoch == 0:
        for name, param in model.named_parameters():
            if 'fc' in name:
                param.requires_grad = True 
            else:
                param.requires_grad =  False
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
        print(f'Total params of model: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')
    elif epoch == 2:
        for param in model.parameters():
            param.requires_grad = True 
        optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    model.train()


Total params of model: 11179077
Training loop is starting with: 5 epoch
Total params of model: 2565


In [38]:
back_bone_params = []
head_param = []

for name, param in model.named_parameters():
    if param.requires_grad:
        if 'fc' in name:
            head_param.append(param)
        else: 
            back_bone_params.append(param)
optimizer = optim.AdamW([
    {'params': back_bone_params, 'lr': 1e-4},
    {'params': head_param, 'lr': 1e-3}
])